# Bideal三段分界线：75/40、80/56与80/55的比较

## tl;dr
80/56在当前搜索中最低误差；80/55为接近的取整选项。

## Context & Methods
### Key Assumptions
固定三段连续直线。对每个候选边界，取真实SOC首次经过95%、上边界、下边界、20%的四个时刻，并连接这四个端点；每块电池都有自己的斜率，但五块电池共用两条SOC边界。
RMSE由整个95→20%区间的真实悬停SOC与该电池自身三段直线的差计算，单位为SOC百分点；总体值为五次飞行MSE的等权平均后开根号。不是将同一条Bideal曲线硬套到五块电池上，也不是把每秒样本当成独立重复实验。
各段至少跨5个百分点、持续10秒；还检查了更宽区间和更长持续时间。边界比较使用相同数据范围和相同计算方法，未改原始SOC、时间或补数据。


## Data
五个同日当前配对；B12只用D5。详细原始来源和哈希见summary.json。

In [1]:
import sys,json
from pathlib import Path
root=Path('/Users/alexmason/Downloads/drone_experiment')
sys.path.insert(0,str(root/'output_py'))
from search_bideal_boundaries import run,OUT
s=run()


Candidates: 1886
 upper  lower   rmse  worst_battery_rmse  min_stage_seconds  top  bottom  min_width  min_seconds  B10_rmse  B11_rmse  B12_rmse  B13_rmse  B14_rmse
    80     56 1.6670              2.2924             29.141   95      20          5           10    1.2679    1.4694    1.4560    1.6590    2.2924
    79     56 1.6821              2.3550             31.150   95      20          5           10    1.2311    1.4225    1.4728    1.7008    2.3550
    81     56 1.6858              2.3025             27.130   95      20          5           10    1.3103    1.5105    1.4434    1.6814    2.3025
    80     54 1.6955              2.3605             29.141   95      20          5           10    1.3264    1.5583    1.3566    1.6652    2.3605
    79     51 1.6974              2.5208             31.150   95      20          5           10    1.2863    1.3784    1.1851    1.7584    2.5208
    79     54 1.6993              2.3860             31.150   95      20          5           10    1

## Results
误差按独立飞行等权，不按采样点数给电池加权。

In [2]:
import pandas as pd, numpy as np
g=pd.read_csv(OUT/'search.csv')
print(g[['upper','lower','rmse']].head(6).to_string(index=False))
print(pd.read_csv(OUT/'sensitivity.csv').to_string(index=False))
assert int(g.iloc[0].upper)==80 and int(g.iloc[0].lower)==56


 upper  lower     rmse
    80     56 1.666997
    79     56 1.682073
    81     56 1.685844
    80     54 1.695462
    79     51 1.697357
    79     54 1.699336
 top  bottom  min_width  min_seconds  upper  lower     rmse  original_rmse
  95      20         10           10     80     56 1.666997       2.504082
  90      20          5           10     79     56 1.502618       2.299318
  95      25          5           10     80     56 1.618999       2.567034
  95      15          5           10     80     56 1.467320       2.449666
  95      11          5           10     80     58 1.716220       2.627707
  95      20          5           15     80     56 1.666997       2.504082


## Takeaways
上述误差是曲线近似误差，不是决策模型准确率，也不是未知飞行的SOC预测误差。旧实验检查和去掉一块电池的检查都重新使用各自真实端点来校准斜率，验证的是分界线的适用性，不是固定耗电率跨飞行迁移的准确性。
当前每个配对只采用一次最新完整实验；搜索最优存在样本内选择偏差，重复实测仍是更强的验证。取整是一种可解释性选择，不是误差最小化本身。
主搜索依据95→20%的实际悬停数据，不据此验证100→95%或20%以下；95→11仅作已有观测范围的敏感性检查，仍不能外推为0%的实测依据。
若采用新边界，需要按新边界重算三段Bideal率和每块电池的转换系数，而不是只改high、medium、low的名称。当前v1及冻结耗电率表保持不变。
